# Preprocessing for the Custom Diffusion Model
- the folder structure is complex and hardcoded, so this follows the structural and naming conventions of the original

In [ ]:
import numpy as np
import shutil
import json
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# Set configs (specified by the Custom Diffusion repo)
split_root      = Path('/path/to/split_root')       
generation_root = Path('/path/to/custom_diffusion/generation')
images_root     = Path('/path/to/custom_diffusion/images')    
ids_dir         = Path('./data/ids/raw')                  
meta_dir        = Path('./data/metadata/dataset')         


images_root.mkdir(parents=True, exist_ok=True)
ids_dir.mkdir(parents=True, exist_ok=True)
meta_dir.mkdir(parents=True, exist_ok=True)


patient_txts = {
    "train": Path('/path/to/train_patients.txt'),
    "val":   Path('/path/to/validation_patients.txt'),
    "test":  Path('/path/to/test_patients.txt'),
}

'''
the target modality is defined as such due to hardcoded specifications: 
Naming files as t1c (which is the actual target modality) is nontrivial, as it would require changing a large portion of the custom diffusion code.
Therefore, png files were named *_flair_*.png, even though they actually contained t1c scans.
'''
MODALITY = "flair"       
TARGET_SIZE   = (224, 160)  # (width, height) for final crop

splits = ["train", "validation", "test"]



## 1. Generate the condition and seg_mod slices datasets

In [ ]:
'''
Normalise the target modality to [0, 255] and copy over seg_mod
Output filenames: {idx}_{MODALITY}_{healthy/unhealthy}.png
                  {idx}_seg_{healthy/unhealthy}.png

These are hardcoded requirements for the model to run propoerly
'''
for split in splits:
    out_split = generation_root / split
    out_split.mkdir(parents=True, exist_ok=True)

    for patient in sorted(p for p in (split_root / split).iterdir() if p.is_dir()):
        
        target_dir  = patient / "t1c"
        seg_mod_dir = patient / "seg_mod"
        out_dir   = out_split / patient.name
        out_dir.mkdir(parents=True, exist_ok=True)

        if not target_dir.exists():
            print(f" {split}/{patient.name} — missing {"t1c"}")
            continue
        if not seg_mod_dir.exists():
            print(f"{split}/{patient.name} — missing seg_mod")
            continue

        target_files    = sorted(target_dir.glob("*.png"))
        seg_mod_files = sorted(seg_mod_dir.glob("*.png"))

        if len(target_files) != len(seg_mod_files):
            print(f"{patient.name} — t1c/seg_mod count mismatch, using minimum")

        for i, (target_file, seg_file) in enumerate(zip(target_files, seg_mod_files)):
            # Normalise image to [0, 255]
            target_img = np.array(Image.open(target_file)).astype("float32")
            if target_img.max() > target_img.min():
                target_norm = (target_img - target_img.min()) / (target_img.max() - target_img.min())
            else:
                target_norm = np.zeros_like(target_img)
            target_norm = (target_norm * 255).astype("uint8")

            # Determine healthy/unhealthy from seg_mod (based on tumor region presence)
            seg_arr = np.array(Image.open(seg_file))
            HEALTHY_VAL = 255   # because it was set to 4 and scaled to 255 earlier)
            label = "unhealthy" if np.any(seg_arr[seg_arr != HEALTHY_VAL] > 0) else "healthy"

            # NOW save the name as "flair" EVEN THOUGH it is actually t1c
            Image.fromarray(target_norm).save(out_dir / f"{i:02d}_{MODALITY}_{label}.png")
            Image.open(seg_file).save(out_dir / f"{i:02d}_seg_{label}.png")   # copy seg_mod as-is

        print(f" {split}/{patient.name} ({len(target_files)} slices)")




## 2. Crop to target size and merge into flat folders

In [ ]:
# Center crop
target_w, target_h = TARGET_SIZE

for split in splits:
    for patient_folder in sorted(p for p in (generation_root / split).iterdir() if p.is_dir()):
        for img_path in patient_folder.glob("*.png"):
            img  = Image.open(img_path)
            w, h = img.size
            left = (w - target_w) // 2
            top  = (h - target_h) // 2
            img.crop((left, top, left + target_w, top + target_h)).save(img_path)
        print(f" {split}/{patient_folder.name}")

# merge into flat folders
for split in splits:
    for patient in sorted(p for p in (generation_root / split).iterdir() if p.is_dir()):
        dst = images_root / patient.name
        if dst.exists():
            print(f" {patient.name} already exists — skipping")
            continue
        shutil.copytree(patient, dst)
        print(f" {patient.name} ← {split}")

## 3. Create TSV files and metadata

In [ ]:

# load patients
def load_patient_dirs(txt_path, base_dir):
    return [base_dir / p.strip() for p in txt_path.read_text().splitlines() if p.strip()]

# create list of patients 
def create_datalist(patient_dirs, base_dir):
    rows = []
    for patient_dir in tqdm(patient_dirs):
        if not patient_dir.exists():
            continue
        for cond_path in sorted(patient_dir.glob(f"*_{MODALITY}_*.png")):
            cond_rel = cond_path.relative_to(base_dir)
            seg_rel  = cond_rel.parent / cond_rel.name.replace(f"_{MODALITY}_", "_seg_")
            label    = cond_path.stem.split("_")[-1]   # "healthy" or "unhealthy"
            rows.append({MODALITY: str(cond_rel), "seg": str(seg_rel), "status": label})
    return pd.DataFrame(rows)

def calculate_statistics(df):
    if df.empty:
        return {"total": 0, "healthy_quantity": 0, "healthy_percentage": 0}
    counts    = df["status"].value_counts()
    healthy   = int(counts.get("healthy", 0))
    unhealthy = int(counts.get("unhealthy", 0))
    total     = healthy + unhealthy
    return {
        "total":              total,
        "healthy_quantity":   healthy,
        "healthy_percentage": round(healthy / total * 100, 2) if total > 0 else 0,
    }

metadata = {"patients": {}}

for split_name, txt_path in patient_txts.items():
    patient_dirs = load_patient_dirs(txt_path, images_root)
    metadata["patients"][split_name] = len(patient_dirs)

    df  = create_datalist(patient_dirs, images_root)
    stats = calculate_statistics(df)

    df.to_csv(ids_dir / f"{split_name}.tsv", sep="\t", index=False)
    metadata[split_name] = stats
    print(f"  [OK] {split_name} — {stats['total']} slices")

# Name is a hardcoded requirement
with open(meta_dir / "02_split_dataset.json", "w") as f:
    json.dump(metadata, f, indent=4)

print(json.dumps(metadata, indent=4))

